# 166 — Token Influence Tracking

Track how each input token influences the final prediction through
attention, residual stream, and per-head paths.

## Why This Matters

Token influence tracking tracks how individual token representations evolve through the network. Understanding token-level dynamics reveals how context is integrated, when predictions form, and how different positions interact to produce the output.

**Key references:**
- [Elhage et al. (2021) "A Mathematical Framework for Transformer Circuits"](https://transformer-circuits.pub/2021/framework/index.html) — Foundational framework for attention head and residual stream analysis

In [ ]:
import jax
import jax.numpy as jnp
from irtk import HookedTransformer, HookedTransformerConfig
from irtk.token_influence_tracking import (
    token_influence_via_attention,
    token_influence_via_residual,
    multi_token_influence,
    influence_path_analysis,
    influence_on_logits,
)

In [ ]:
cfg = HookedTransformerConfig(
    n_layers=4, d_model=32, n_ctx=64, d_head=8, n_heads=4, d_vocab=100,
)
model = HookedTransformer(cfg)
key = jax.random.PRNGKey(42)
leaves, treedef = jax.tree.flatten(model)
new_leaves = []
for leaf in leaves:
    if isinstance(leaf, jnp.ndarray) and leaf.dtype == jnp.float32:
        key, subkey = jax.random.split(key)
        new_leaves.append(jax.random.normal(subkey, leaf.shape) * 0.1)
    else:
        new_leaves.append(leaf)
model = jax.tree.unflatten(treedef, new_leaves)
tokens = jnp.arange(15)

In [ ]:
result = token_influence_via_attention(model, tokens, source_pos=0)
for p in result['per_layer']:
    heads = ', '.join(f"H{h['head']}={h['attention']:.3f}" for h in p['per_head_attention'])
    print(f"Layer {p['layer']}: mean_attn={p['direct_attention']:.4f}  "
          f"dominant=H{p['dominant_head']}  [{heads}]")

In [ ]:
result = token_influence_via_residual(model, tokens, source_pos=0)
for p in result['per_layer']:
    bar = '#' * int(abs(p['influence']) * 20)
    sign = '+' if p['influence'] > 0 else '-'
    print(f"Layer {p['layer']}: cos={p['influence']:+.4f}  mag={p['magnitude']:+.4f}  {sign}{bar}")

In [ ]:
result = multi_token_influence(model, tokens)
print(f"Most influential: pos {result['most_influential']}  "
      f"entropy={result['influence_entropy']:.3f} (norm={result['normalized_entropy']:.3f})\n")
for p in result['per_position']:
    bar = '#' * int(p['influence'] * 50)
    print(f"  pos {p['position']:2d}: {p['influence']:.4f}  {bar}")

In [ ]:
result = influence_path_analysis(model, tokens, source_pos=0)
for p in result['per_layer']:
    paths = sorted(p['head_paths'], key=lambda x: -x['path_strength'])
    top = paths[0]
    print(f"Layer {p['layer']}: strongest=H{top['head']} "
          f"(attn={top['attention_to_source']:.3f} norm={top['output_norm']:.3f} "
          f"strength={top['path_strength']:.4f})")

In [ ]:
result = influence_on_logits(model, tokens, source_pos=0, top_k=5)
print(f"Total influence norm: {result['total_logit_change_norm']:.4f}\n")
print('Promoted:', ', '.join(f"tok{t['token']}({t['logit_change']:+.3f})" for t in result['top_promoted']))
print('Demoted:', ', '.join(f"tok{t['token']}({t['logit_change']:+.3f})" for t in result['top_demoted']))